In [ ]:
using Pkg
Pkg.activate(joinpath(dirname(dirname(@__DIR__)), "em-master"))

script_dir = dirname(dirname(@__DIR__))
em_dir = joinpath(script_dir, "em-master")
study_file = joinpath(script_dir, "Data", "Processed", "Study2.csv")

full = false              #set this to true when running study 1 to help with convergence
emtol = 1e-3
push!(LOAD_PATH, em_dir)

using EM, Distributions, LinearAlgebra, Optim, Statistics, Random
using GLM, DataFrames, NamedArrays, SpecialFunctions, CSV

data = CSV.read(study_file, DataFrame)
rename!(data, :subn => :sub)

unique_subjects = unique(data.sub)
subject_map = Dict(unique_subjects[i] => i for i in 1:length(unique_subjects))
data.sub .= [subject_map[sub] for sub in data.sub]

NS = 57
subs = 1:NS

X = ones(Int, NS)
startbetas = [0 0 0 0 0 0 0 0 0]
startsigma = [1, 1, 1, 1, 1, 1, 1, 1, 1]

function S1_Interaction(params, data)

    AS_C_T = params[1]
    AD_C_T = params[2]
    AS_U_T = params[3]
    d_Int_T = params[4]

    AS_C_F = params[5]
    AD_C_F = params[6]
    AS_U_F = params[7]
    d_Int_F = params[8]

    AD_U_T = AD_C_T + d_Int_T
    AD_U_F = AD_C_F + d_Int_F

    aPC_T = 1 / (1 + exp(-0.5 * (AS_C_T + AD_C_T)))
    aNC_T = 1 / (1 + exp(-0.5 * (AS_C_T - AD_C_T)))
    aPU_T = 1 / (1 + exp(-0.5 * (AS_U_T + AD_U_T)))
    aNU_T = 1 / (1 + exp(-0.5 * (AS_U_T - AD_U_T)))

    aPC_F = 1 / (1 + exp(-0.5 * (AS_C_F + AD_C_F)))
    aNC_F = 1 / (1 + exp(-0.5 * (AS_C_F - AD_C_F)))
    aPU_F = 1 / (1 + exp(-0.5 * (AS_U_F + AD_U_F)))
    aNU_F = 1 / (1 + exp(-0.5 * (AS_U_F - AD_U_F)))

    Beta = 50 / (1 + exp(-params[9]))

    R, C1, C2, TrialType, FeedbackType, Condition = (
        data.Reward, data.C1, data.C2, data.TrialType,
        data.FeedbackType, data.Condition
    )

    Q = zeros(typeof(Beta), 2, 4)
    lik = 0.0

    for t in eachindex(C1)
        QC = Condition[t]
        chosen = C1[t] == 1 ? 1 : 2
        unchosen = 3 - chosen

        lik += log(1 / (1 + exp(-Beta * (Q[chosen, QC] - Q[unchosen, QC]))))

        is_chosen_feedback = FeedbackType[t] == 1
        pe = is_chosen_feedback ? (R[t] - Q[chosen, QC]) : (R[t] - Q[unchosen, QC])

        if is_chosen_feedback
            if TrialType[t] == 1
                lr = pe > 0 ? aPC_T : aNC_T
            else
                lr = pe > 0 ? aPC_F : aNC_F
            end
            Q[chosen, QC] = (1 - lr) * Q[chosen, QC] + lr * R[t]
        else
            if TrialType[t] == 1
                lr = pe > 0 ? aPU_T : aNU_T
            else
                lr = pe > 0 ? aPU_F : aNU_F
            end
            Q[unchosen, QC] = (1 - lr) * Q[unchosen, QC] + lr * R[t]
        end
    end

    return -lik
end

(betas, sigma, x, l, h) = em(data, subs, X, startbetas, startsigma, S1_Interaction, maxiter=2000, emtol=emtol, full=full)

In [ ]:
using Distributions

standarderrors = emerrors(data, subs, x, X, h, betas, sigma, S1_Interaction)

function em_ttest(betas, standarderrors, m; d=nothing, nsubjects=nothing)
    betas = vec(betas)
    if d === nothing d = 1:length(betas) end
    if length(m) != length(d) error("Mismatch: m has $(length(m)) elements, but $(length(d)) parameters.") end

    se = standarderrors[1]
    a = betas[d]
    e = se[d]
    df = nsubjects - 1

    t = (a .- m) ./ e
    p = 2 * [ccdf(TDist(df), abs(ti)) for ti in t]

    stats = (tstat = t, pval = p, df = df)
    return stats
end

m = zeros(2)

d = [4, 8]

stats = em_ttest(betas, standarderrors, m; d=d, nsubjects=length(subs))

println(stats)